In [ ]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_excel('Excels/ADOP_Limpio.xlsx') 

df['CONTRASTE'] = pd.to_numeric(df['CONTRASTE'], errors='coerce').fillna(99).astype(int)
df['PUESTO'] = pd.to_numeric(df['PUESTO'], errors='coerce').fillna(99).astype(int)

diccionario_niveles = {'NO': 0, 'PRO': 1, 'ELITE': 2, 'BRONCE': 3, 'PLATA': 4, 'ORO': 5}
diccionario_inverso = {0: 'NO', 1: 'PRO', 2: 'ELITE', 3: 'BRONCE', 4: 'PLATA', 5: 'ORO'}

df['NIVEL'] = df['NIVEL'].map(diccionario_niveles).fillna(0).astype(int)
df['PROPUESTA'] = df['PROPUESTA'].map(diccionario_niveles).fillna(0).astype(int)

In [ ]:
FEATURES = ['PUESTO', 'NIVEL', 'CONTRASTE', 'PARTICIPACION', 'TIENE_HISTORICO', 'TENDENCIA']
X = df[FEATURES]
y = df['PROPUESTA']

modelo_final = RandomForestClassifier(
    n_estimators=150,
    max_depth=4,
    class_weight='balanced',
    random_state=42
)

clases_presentes = np.unique(y)
nombres_clases = [f"{diccionario_inverso[c]}" for c in clases_presentes]

In [ ]:
cv_oscar = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(modelo_final, X, y, cv=cv_oscar)

print("="*60 + "\n")
print("Generando Matriz de Confusión visual...")

cm = confusion_matrix(y, y_pred_cv)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', linewidths=1, linecolor='white',
            xticklabels=nombres_clases, yticklabels=nombres_clases)

plt.title('Matriz de Confusión: Cross-Validation 5 Folds', fontsize=14, pad=15)
plt.xlabel('Propuesta Sugerida por el Modelo', fontsize=12, labelpad=10)
plt.ylabel('Propuesta Real (Histórico)', fontsize=12, labelpad=10)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

reporte_texto = classification_report(y, y_pred_cv, target_names=nombres_clases, zero_division=0)
print(reporte_texto)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
modelo_final.fit(X_train, y_train)

predicciones_numericas = modelo_final.predict(X)

df['PREDICCION_IA'] = [diccionario_inverso[num] for num in predicciones_numericas]
df['PROPUESTA'] = df['PROPUESTA'].map(diccionario_inverso)
df['NIVEL'] = df['NIVEL'].map(diccionario_inverso)

ruta_excel = 'Predicciones_ADOP_2026.xlsx'

from openpyxl.styles import PatternFill, Font, Alignment
with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:
    df.to_excel(writer, index=False, sheet_name='Resultados_IA')
    
    worksheet = writer.sheets['Resultados_IA']
    
    header_fill = PatternFill(start_color='002060', end_color='002060', fill_type='solid')
    header_font = Font(color='FFFFFF', bold=True)
    
    for cell in worksheet[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center')
        
    for column in worksheet.columns:
        max_length = 0
        column_letter = column[0].column_letter
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        worksheet.column_dimensions[column_letter].width = max_length + 2


joblib.dump(modelo_final, 'Interfaz/modelo_adop.pkl')
print("Cerebro guardado como 'modelo_adop.pkl'. Listo para la web.")